# TensorBoard Viewer

Independent notebook with its own kernel; the training notebook `autoTest.ipynb` is never blocked. Use Runtime -> Change runtime type -> **CPU** to save GPU quota.

**Design idea:** the iframe is bound to **port 6006** via `output.serve_kernel_port_as_iframe`, NOT to the underlying TB process. That means we can kill and restart TB freely; as long as the new process binds to the same port, the existing iframe auto-reconnects without moving on the page. You only re-run the small refresh cell, and the iframe stays put.

## 1. Mount Google Drive (run once per session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Launch TensorBoard (run once per session)
Defines `refresh_tb()` and runs it once: rsync Drive -> local SSD, then start a fresh TB process on port 6006. The iframe is created with `serve_kernel_port_as_iframe(port=6006)` so it survives later restarts of the TB process.

**Do not re-run this cell** during the session unless the iframe itself broke. Use cell 3 below to refresh data.

In [ ]:
import subprocess
import sys
import time
from google.colab import output

SRC = '/content/drive/MyDrive/autoTest/models/stage1_transformer/tensorboard/'
DST = '/content/local_tb/'
TB_PORT = 6006


def _run(cmd, shell=False):
    """Run a subprocess and write its captured output to the notebook cell.

    Plain subprocess.run() inherits the kernel's raw FD 1, which Jupyter does
    not route to the cell output. Capturing then writing via sys.stdout works.
    """
    result = subprocess.run(
        cmd, shell=shell, capture_output=True, text=True, check=False,
    )
    if result.stdout:
        sys.stdout.write(result.stdout)
    if result.stderr:
        sys.stdout.write(result.stderr)
    return result.returncode


def refresh_tb():
    """rsync Drive -> local SSD, then restart TensorBoard on TB_PORT.

    The iframe (created once via serve_kernel_port_as_iframe) reconnects
    automatically once the new TB process is listening on TB_PORT.
    """
    _run(['mkdir', '-p', DST])

    # Force Drive FUSE to refresh metadata for every event file.
    print('--- Drive event files ---')
    _run(
        f"find {SRC} -type f -name 'events.out.tfevents.*' "
        f"-printf '%s\\t%p\\n' | sort",
        shell=True,
    )

    # Incremental copy with --inplace so file inodes stay stable.
    print('\n--- rsync ---')
    _run(['rsync', '-av', '--inplace', SRC, DST])

    print('\n--- local event files after sync ---')
    _run(
        f"find {DST} -type f -name 'events.out.tfevents.*' "
        f"-printf '%s\\t%p\\n' | sort",
        shell=True,
    )

    # Kill any existing TB process so we get a fresh one with fresh file state.
    print('\n--- restarting TB ---')
    _run(['pkill', '-f', 'tensorboard'])
    time.sleep(2)

    # Start new TB on TB_PORT, bound to all interfaces so the Colab kernel proxy
    # can reach it. stdout/stderr go to /tmp/tb.log so the cell output stays clean.
    log = open('/tmp/tb.log', 'a')
    subprocess.Popen(
        [
            'tensorboard',
            '--logdir', DST,
            '--port', str(TB_PORT),
            '--bind_all',
            '--reload_interval', '5',
            '--reload_multifile', 'true',
        ],
        stdout=log, stderr=subprocess.STDOUT,
    )

    # Wait until TB is listening (up to ~10 s).
    for _ in range(20):
        check = subprocess.run(
            ['bash', '-c', f'ss -ltn | grep -q :{TB_PORT}'],
            check=False,
        )
        if check.returncode == 0:
            print(f'TB ready on port {TB_PORT}')
            return
        time.sleep(0.5)
    print(f'WARNING: TB did not bind to port {TB_PORT} within 10s; check /tmp/tb.log')


# Initial start.
refresh_tb()

# Print the proxy URL so it is easy to verify / open in a new tab.
proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({TB_PORT})')
print(f'\nTB proxy URL: {proxy_url}')

# Embed the iframe bound to TB_PORT. This iframe persists across later
# refresh_tb() calls because it is tied to the port, not to the TB process.
output.serve_kernel_port_as_iframe(TB_PORT, height='800', cache_in_notebook=False)

## 3. Refresh data (re-run this cell to see new training progress)
Re-runs `refresh_tb()`: rsync Drive -> local SSD, kill TB, restart TB on the same port. The iframe in cell 2 stays put and reconnects automatically.

After this cell finishes (a few seconds), scroll back to cell 2's iframe and click its refresh button (or wait ~5 s for TB's auto-poll).

In [ ]:
refresh_tb()
print(f"\nrefreshed at {time.strftime('%H:%M:%S')}")